# RAG Chatbot with BioBERT and LLaMA 2

This notebook implements a Retrieval-Augmented Generation (RAG) pipeline using biomedical embeddings (`BioBERT`) and a local language model (`LLaMA 2`) to answer domain-specific medical questions. The pipeline includes:

- Document loading and splitting  
- Embedding generation with `BioBERT`  
- Vector store creation using `Chroma`  
- Retrieval-based question answering with `LLaMA 2`  

This setup is ideal for medical QA tasks where factual grounding and domain relevance are critical.  
The model runs locally via `transformers` and Hugging Face pipelines, without requiring an API key.

---

In [1]:
!pip install -U langchain langchain-community langchain-huggingface
!pip install -U chromadb
!pip install -U sentence-transformers
!pip install -U transformers
!pip install -U accelerate
!pip install -U langchain-chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.2 MB/s eta 0:00:00


In [2]:
import os
import glob
import shutil

from huggingface_hub import login

from langchain.schema import Document
from langchain.document_loaders import WikipediaLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

### Project Directory Connection


In [3]:
!git clone https://github.com/a20190202/PLN_Medical_Flashcard.git

Cloning into 'PLN_Medical_Flashcard'...
remote: Enumerating objects: 405, done.
remote: Counting objects: 100% (405/405), done.
remote: Compressing objects: 100% (275/275), done.
remote: Total 405 (delta 167), reused 339 (delta 109), pack-reused 0 (from 0)
Receiving objects: 100% (405/405), 38.34 MiB | 10.41 MiB/s, done.
Resolving deltas: 100% (167/167), done.
Updating files: 100% (83/83), done.


In [4]:
!ls

PLN_Medical_Flashcard  sample_data


In [5]:
%cd PLN_Medical_Flashcard/pln_model

/content/PLN_Medical_Flashcard/pln_model


In [6]:
!pwd

/content/PLN_Medical_Flashcard/pln_model


# Vectorstore Generation
---

In [7]:
def read_txt_files(folder_path):
    all_docs = []
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=100,
        length_function=len,
    )

    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            file_path = os.path.join(folder_path, filename)
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()

            chunks = splitter.split_text(text)

            for i, chunk in enumerate(chunks):
                doc = Document(
                    page_content=chunk,
                    metadata={
                        "source": filename,
                        "chunk_id": i,
                        "total_chunks": len(chunks)
                    }
                )
                all_docs.append(doc)

    return all_docs

all_documents = read_txt_files("data/textbooks")

## Embeddings model  
### `pritamdeka/S-Biomed-Roberta-snli-multinli-stsb`

This SentenceTransformer model is a fine-tuned version of `allenai/biomed_roberta_base` trained on multiple natural language inference (NLI) and semantic similarity datasets, including:

- SNLI, MNLI, and STS-B

It is optimized for sentence-level semantic similarity tasks in the biomedical domain, making it well-suited for generating dense embeddings of medical questions, abstracts, or documents for use in retrieval pipelines.

- **Base model:** `allenai/biomed_roberta_base`  
- **Embedding dimensions:** `768`  
- **Use case:** Biomedical sentence embeddings for retrieval, similarity search, and clustering

Model link: [https://huggingface.co/pritamdeka/S-Biomed-Roberta-snli-multinli-stsb](https://huggingface.co/pritamdeka/S-Biomed-Roberta-snli-multinli-stsb)


In [8]:
EMBEDDINGS = "pritamdeka/S-Biomed-Roberta-snli-multinli-stsb"

In [9]:
embeddings_model = HuggingFaceEmbeddings(
        model_name=EMBEDDINGS,
    )

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
# Create Vector Store (Run Only Once)
vectorstore = Chroma.from_documents(
    documents=all_documents,
    embedding=embeddings_model,
    persist_directory=f"./{EMBEDDINGS.replace('/','_')}"
)

### Save the vector store after creation

In [ ]:
# Define original path (correct one where Chroma actually saved the files)
vectorstore_dir = f"./{EMBEDDINGS.replace('/', '_')}"
zip_path = f"{vectorstore_dir}.zip"

# Create the ZIP file from the original directory
shutil.make_archive(vectorstore_dir, 'zip', vectorstore_dir)

# Move the ZIP to /content so it's visible in Colab file browser
!mv "{zip_path}" /content/

print(f"Vector store zipped and moved to /content/: {os.path.basename(zip_path)}")

Vector store zipped and moved to /content/: pritamdeka_S-Biomed-Roberta-snli-multinli-stsb.zip


### Load the saved vector store

In [ ]:
# Unzip the saved vector store
vectorstore_dir = f"./{EMBEDDINGS.replace('/', '_')}"
zip_path = f"/content/{EMBEDDINGS.replace('/', '_')}.zip"

# Unzip only if not already extracted
if not os.path.exists(vectorstore_dir):
    shutil.unpack_archive(zip_path, vectorstore_dir)
    print(f"✅ Unzipped vector store to: {vectorstore_dir}")
else:
    print(f"ℹ️ Directory already exists: {vectorstore_dir}")

# Load the vector store
vectorstore = Chroma(
    persist_directory=vectorstore_dir,
    embedding_function=embeddings_model
)

ℹ️ Directory already exists: ./pritamdeka_S-Biomed-Roberta-snli-multinli-stsb


# RAG
---

In [11]:
# OLlama download
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [12]:
# Launch the Ollama server locally
!ollama serve > /dev/null 2>&1 &
!sleep 10

In [13]:
!ollama pull llama2:latest
!pip install -U langchain-ollama

In [14]:
from langchain_ollama import OllamaLLM

In [15]:
llm = OllamaLLM(
    model="llama2:latest",
    temperature=0.05,
    system=""
)

## Personalized prompt:

In [16]:
prompt = ChatPromptTemplate.from_template(
"""
Medical Flashcard Generator Prompt

You will receive the name of a medical condition or disease. Your task is to create 5 comprehensive flashcards that systematically cover the essential aspects of the condition for medical education purposes.

Required Coverage Areas:
1. Definition & Pathophysiology - Core concept and underlying mechanisms
2. Etiology & Risk Factors - Causes and predisposing factors
3. Clinical Presentation - Signs, symptoms, and clinical manifestations
4. Diagnostic Approach - Key tests, criteria, and differential considerations
5. Management & Treatment - Therapeutic interventions and prognosis

Flashcard Requirements:
- Each flashcard must contain one focused question and one comprehensive answer
- Questions should be clinically relevant and test practical knowledge
- Answers should be precise, direct, and medically accurate
- Provide specific details (lab values, medication dosages, timeframes where applicable)
- Use medical terminology appropriately while maintaining clarity
- Give concise, focused responses without bullet points or lists
- Prioritize high-yield information commonly tested in medical examinations

Context Considerations:
{context}

Output Format:
Flashcard 1: [Topic Area]
Q: [Specific, focused question]
A: [Precise, direct answer in paragraph form]

Flashcard 2: [Topic Area]
Q: [Specific, focused question]
A: [Precise, direct answer in paragraph form]

[Continue for all 5 flashcards]

Quality Standards:
- Ensure medical accuracy and evidence-based content
- Use current clinical guidelines and best practices
- Include relevant mnemonics or memory aids where helpful
- Maintain consistency in terminology and formatting
- Focus on clinically actionable information

Medical Condition: {input}
"""
 )

## RAG Pipeline:

In [17]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# RAG Chain
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(retriever, combine_docs_chain)

## Inference Test:

In [19]:
import time

### __Diabetes:__

In [20]:
question = "Diabetes"

t0 = time.time()
result = retrieval_chain.invoke({
    "input": question
})
total_time = time.time() - t0

print("Respuesta:", result["answer"])
print("\nTotal time:", total_time)

Respuesta: Flashcard 1: Definition & Pathophysiology of Diabetes
Q: What is diabetes mellitus?
A: Diabetes mellitus is a chronic metabolic disorder characterized by high blood glucose levels due to insulin deficiency or insulin resistance. The underlying mechanisms include impaired insulin secretion, insulin resistance, and defective glucose metabolism.

Flashcard 2: Etiology & Risk Factors of Diabetes
Q: What are the main causes of diabetes?
A: The main causes of diabetes include genetic predisposition, obesity, physical inactivity, and an unhealthy diet. Other risk factors include older age, family history, and certain ethnicities.

Flashcard 3: Clinical Presentation of Diabetes
Q: What are the common symptoms of diabetes?
A: The common symptoms of diabetes include increased thirst and urination, fatigue, blurred vision, slow healing of cuts and wounds, and recurring skin, gum, or bladder infections.

Flashcard 4: Diagnostic Approach to Diabetes
Q: How is diabetes diagnosed?
A: Diabe

### __Asthma:__

In [21]:
question = "Asthma"

t0 = time.time()
result = retrieval_chain.invoke({
    "input": question
})
total_time = time.time() - t0

print("Respuesta:", result["answer"])
print("\nTotal time:", total_time)

Respuesta: Flashcard 1: Definition & Pathophysiology
Q: What is asthma?
A: Asthma is a chronic inflammatory disorder of the airways characterized by recurrent episodes of wheezing, breathlessness, chest tightness, and cough, particularly at night and/or early in the morning. The hallmarks of asthma are intermittent, reversible airway obstruction; chronic bronchial inflammation with eosinophils; bronchial smooth muscle cell hypertrophy and hyperreactivity; and increased mucus secretion.

Flashcard 2: Etiology & Risk Factors
Q: What are the risk factors for developing asthma?
A: The etiology of asthma is complex, involving both genetic and environmental factors. Susceptibility to asthma is inherited in an autosomal dominant pattern, with polymorphism genes on chromosomes 5q that include cytokine gene clusters, 3-adrenergic and glucocorticoid receptor genes, and the T-cell antigen receptor gene. Environmental allergic stimuli such as influenza or cigarette smoke can serve as a promoter fo

### __Cardiac Arrest:__

In [22]:
question = "Cardiac Arrest"

t0 = time.time()
result = retrieval_chain.invoke({
    "input": question
})
total_time = time.time() - t0

print("Respuesta:", result["answer"])
print("\nTotal time:", total_time)

Respuesta: Flashcard 1: Definition & Pathophysiology of Cardiac Arrest
Q: What is cardiac arrest?
A: Cardiac arrest is a sudden loss of cardiac function leading to cessation of breathing and circulation, usually due to a lethal arrhythmia such as ventricular fibrillation or pulseless ventricular tachycardia. The underlying mechanisms involve a failure of the heart's electrical conduction system, resulting in a stop in blood flow and oxygen delivery to vital organs.

Flashcard 2: Etiology & Risk Factors of Cardiac Arrest
Q: What are the common causes of cardiac arrest?
A: The most common cause of cardiac arrest is coronary artery disease, followed by hypertrophic or dilated cardiomyopathy, and inherited ion channelopathies such as long QT syndrome or Brugada syndrome. Other risk factors include older age, male gender, smoking, alcohol consumption, and a history of previous cardiac events.

Flashcard 3: Clinical Presentation of Cardiac Arrest
Q: What are the signs and symptoms of cardiac

### __Gastritis:__

In [23]:
question = "Gastritis"

t0 = time.time()
result = retrieval_chain.invoke({
    "input": question
})
total_time = time.time() - t0

print("Respuesta:", result["answer"])
print("\nTotal time:", total_time)

Respuesta: Flashcard 1: Definition & Pathophysiology of Gastritis
Q: What is gastritis?
A: Gastritis is a term used to describe inflammation of the stomach mucosa, characterized by the presence of subepithelial hemorrhages and erosions. The underlying mechanisms include infection with Helicobacter pylori, nonsteroidal anti-inflammatory drug (NSAID) use, alcohol consumption, and stress.

Flashcard 2: Etiology & Risk Factors of Gastritis
Q: What are the common causes of gastritis?
A: The most common causes of acute gastritis are infectious agents such as Helicobacter pylori, while chronic gastritis can be caused by NSAID use, alcohol consumption, and stress. Other risk factors include smoking, obesity, and a poor diet.

Flashcard 3: Clinical Presentation of Gastritis
Q: What are the clinical manifestations of gastritis?
A: The clinical presentation of gastritis can vary depending on the severity of the inflammation. Common symptoms include epigastric pain, nausea, vomiting, and abdominal

### __Stroke:__

In [24]:
question = "Stroke"

t0 = time.time()
result = retrieval_chain.invoke({
    "input": question
})
total_time = time.time() - t0

print("Respuesta:", result["answer"])
print("\nTotal time:", total_time)

Respuesta: Flashcard 1: Stroke
Q: What are the core concepts and underlying mechanisms of stroke?
A: Stroke is a neurological disorder caused by the interruption of blood flow to the brain, leading to cell death and tissue damage. The core concepts include cerebral vasculature, blood clotting, and hemorrhage. Underlying mechanisms involve platelet activation, coagulation cascades, and vascular obstruction or rupture.

Flashcard 2: Stroke
Q: What are the common causes of stroke?
A: The most common causes of stroke include atherosclerosis, hypertension, heart disease, and blood clotting disorders. Other risk factors include diabetes, smoking, obesity, and lack of physical activity.

Flashcard 3: Stroke
Q: What are the clinical presentation and diagnosis of stroke?
A: Clinical presentation may include sudden weakness or numbness in the face, arm, or leg, difficulty speaking or understanding speech, sudden vision loss, dizziness, or loss of balance, and severe headache. Diagnosis is based 

In [25]:
question = "Glaucoma"

t0 = time.time()
result = retrieval_chain.invoke({
    "input": question
})
total_time = time.time() - t0

print("Respuesta:", result["answer"])
print("\nTotal time:", total_time)

Respuesta: Flashcard 1: Glaucoma
Q: What is the core concept and underlying mechanisms of glaucoma?
A: Glaucoma is a slowly progressive, insidious optic neuropathy associated with chronic elevation of intraocular pressure (IOP). The underlying mechanism involves damage to the optic nerve fibers due to increased pressure, leading to atrophy and loss of retinal ganglion cells. This results in visual field defects and eventual blindness if left untreated.

Flashcard 2: Glaucoma - Etiology & Risk Factors
Q: What are the main causes and predisposing factors for glaucoma?
A: The main causes of glaucoma are chronic elevation of IOP, which can be due to various factors such as ocular hypertension, angle closure, or exfoliation. Risk factors include age (especially over 60), family history, diabetes, hypertension, and myopia.

Flashcard 3: Glaucoma - Clinical Presentation
Q: What are the clinical manifestations of glaucoma?
A: The clinical presentation of glaucoma includes visual field defects 

In [26]:
question = "Epilepsy"

t0 = time.time()
result = retrieval_chain.invoke({
    "input": question
})
total_time = time.time() - t0

print("Respuesta:", result["answer"])
print("\nTotal time:", total_time)

Respuesta: Flashcard 1: Definition & Pathophysiology
Q: What is epilepsy?
A: Epilepsy is a chronic neurological disorder characterized by recurrent seizures resulting from abnormal electrical activity in the brain. The underlying mechanisms involve alterations in neural excitability, synaptic plasticity, and brain network organization.

Flashcard 2: Etiology & Risk Factors
Q: What are the common causes of epilepsy?
A: The most common causes of epilepsy include genetic mutations, head trauma, infections, stroke or bleeding in the brain, and developmental disorders. Risk factors for developing epilepsy include family history, age (especially childhood), and certain medical conditions.

Flashcard 3: Clinical Presentation
Q: What are the clinical manifestations of epilepsy?
A: The clinical presentation of epilepsy varies depending on the type of seizure disorder, but can include sudden loss of consciousness, convulsions, muscle stiffness or spasms, and altered mental status. Seizures can a

In [27]:
question = "Pneumonia"

t0 = time.time()
result = retrieval_chain.invoke({
    "input": question
})
total_time = time.time() - t0

print("Respuesta:", result["answer"])
print("\nTotal time:", total_time)

Respuesta: Flashcard 1: Definition & Pathophysiology of Pneumonia
Q: What is pneumonia?
A: Pneumonia is an infection of the lower respiratory tract that involves the airways and parenchyma with consolidation of the alveolar spaces. It can have multiple causes, including viral and bacterial infections or aspiration. The term "lower respiratory tract infection" is often used to encompass bronchitis, bronchiolitis, pneumonia, or any combination of the three.

Flashcard 2: Etiology & Risk Factors of Pneumonia
Q: What are the common causes of pneumonia?
A: Common causes of pneumonia include viral infections such as influenza A or B, parainfluenza virus, and respiratory syncytial virus. Other causes include bacterial infections such as Streptococcus pneumoniae, Haemophilus influenzae, and Staphylococcus aureus. Risk factors for developing pneumonia include age, immunocompromise, chronic medical conditions, and exposure to smoky environments.

Flashcard 3: Clinical Presentation of Pneumonia
Q

In [28]:
question = "Hypothyroidism"

t0 = time.time()
result = retrieval_chain.invoke({
    "input": question
})
total_time = time.time() - t0

print("Respuesta:", result["answer"])
print("\nTotal time:", total_time)

Respuesta: Flashcard 1: Definition & Pathophysiology of Hypothyroidism
Q: What is hypothyroidism?
A: Hypothyroidism refers to a condition where the thyroid gland does not produce enough thyroid hormones, leading to a range of clinical manifestations. The underlying mechanisms involve an insufficiency of T4 and T3 production, which can occur due to various causes such as autoimmune destruction (Hashimoto's thyroiditis), iodine deficiency, surgical removal or radioablation of the thyroid gland, and certain medications.

Flashcard 2: Etiology & Risk Factors of Hypothyroidism
Q: What are the common causes of hypothyroidism?
A: The most common causes of hypothyroidism include autoimmune destruction (Hashimoto's thyroiditis), iodine deficiency, surgical removal or radioablation of the thyroid gland, and certain medications. Other risk factors include family history, age, and gender.

Flashcard 3: Clinical Presentation of Hypothyroidism
Q: What are the clinical manifestations of hypothyroidis

In [29]:
question = "Hepatitis B"

t0 = time.time()
result = retrieval_chain.invoke({
    "input": question
})
total_time = time.time() - t0

print("Respuesta:", result["answer"])
print("\nTotal time:", total_time)

Respuesta: Flashcard 1: Hepatitis B
Q: What are the core concepts and underlying mechanisms of hepatitis B?
A: Hepatitis B is caused by the hepatitis B virus (HBV), which attacks the liver and causes inflammation. The virus replicates in the liver, leading to cell damage and inflammation. HBV can cause acute or chronic infection, with chronic infection being more common. Chronic HBV infection can lead to liver cirrhosis or hepatocellular carcinoma (HCC).

Flashcard 2: Hepatitis B
Q: What are the risk factors for hepatitis B transmission?
A: Hepatitis B is primarily transmitted through percutaneous exposure to infected blood or body fluids, such as through sharing needles or receiving a tattoo with unsterilized equipment. Sexual contact and mother-to-child transmission during childbirth are also common modes of transmission.

Flashcard 3: Hepatitis B
Q: What are the clinical presentation and diagnostic approach for hepatitis B?
A: Acute HBV infection typically presents with symptoms suc